# Testing Claim 1 on an open-data CPU surrogate

## 0. Motivation

The paper studies frozen medical foundation-model embeddings and reports a quantum-support-vector-machine (QSVM) advantage for detecting the minority class. The primary metric is minority-class F1: it is high only when the model finds minority examples with both useful precision and useful recall.

This notebook asks a narrower question: **how sensitive is the local CPU-only PneumoniaMNIST surrogate to two audited protocol choices?** It compares leaky versus train-only MinMax fitting and square-only versus consistent train-trace kernel scaling.

MerLin is included as a fourth, data-paired model. It uses the same N=500 subset, split seed, and PCA dimension as the other models. However, its photonic feature map is not the paper's qubit BSP map, so it is discussed separately as an adaptation rather than as evidence for Claim 1.

This notebook loads small committed result artifacts only. It does not recompute kernels, train models, regenerate figures, or include medical images.

> This notebook reads the committed protocol-matrix artifacts. It does not recompute kernels, train models, or rerun the experiments.

## 1. Claim 1 in the paper

On its MIMIC-CXR insurance task and frozen medical embeddings, the [paper](https://arxiv.org/abs/2604.24597v1) reports that:

- a fidelity/BSP QSVM with `C=1`, one circuit repetition, and trace normalization has higher minority-class F1 than an equally untuned linear SVM with `C=1` for all 18 model–qubit pairs actually tested, with qubit counts drawn from `{4, 6, 8, 9, 10, 11, 12, 16}`;
- across ten embedding seeds, 17 paired comparisons have `p<0.001` and one has `p<0.01` in the paper's paired bootstrap analysis;
- the linear SVM has minority-class F1 equal to zero on 90–100% of seeds at every tested qubit count;
- against the best validation-tuned RBF SVM at the same PCA dimension, the QSVM wins all seven reported configurations, with a reported mean gain of `+0.068`.

A configuration-level win compares aggregate results across seeds; it does not necessarily mean that QSVM wins every individual seed. Our local checks are therefore:

1. Is the mean paired difference `F1(QSVM) - F1(baseline)` positive at q=4 and q=6?
2. How often does QSVM win, tie, or lose for the same local seed?
3. Does the local linear SVM collapse to minority F1 equal to zero?

The paper's seeds generate different embeddings. Our seeds instead control subsampling and the train/validation/test split, so the two protocols are not statistically equivalent.

MerLin does not enter these Claim 1 checks. Section 4 asks a separate descriptive question: how does a matched photonic fidelity kernel behave under the same local data protocol?

## 2. Our setup

| Item | Local adaptation |
|---|---|
| Reference reproduction | Not run; gated MIMIC-CXR embeddings and the insurance target are unavailable locally |
| Scope | Open-data CPU surrogate, not the reference reproduction |
| Dataset | Official PneumoniaMNIST training split |
| Representation | Raw flattened 28×28 pixels, not frozen embeddings |
| Sample count | N=500 for each seed |
| Split | 400 train / 50 validation / 50 test, stratified |
| Minority class | `normal`, with 11–14 test examples depending on the seed |
| Seeds | 0–9, controlling both subsampling and splitting |
| Dimensions | PCA dimension q=4 and q=6; the QSVM uses the same number of qubits |
| QSVM | Upstream-derived BSP path, `C=1`, one repetition, CPU, two trace protocols |
| Linear baseline | Linear SVM, `C=1` |
| RBF baseline | `C` selected from `{0.01, 0.1, 1, 10, 100}` using validation minority F1 |
| MerLin adaptation | Photonic fidelity kernel, `C=1`, exact CPU simulation, fixed circuit seed 0 |
| Primary metric | Test F1 for the `normal` class |

A comparison is data-paired when the protocol, q, and data/split seed match. The completed matrix contains four protocol IDs, two q values, two quantum models, two classical baselines, and 32 aggregate rows over ten paired seeds. MerLin's circuit seed remains fixed at 0.

The main committed input is `results/protocol_matrix_n500_q4_q6/protocol_summary.csv`. `protocol_results_per_seed.csv` is available for paired detail but is not needed by the aggregate tables below. `results/q4_q6_n500_per_seed.csv` is retained only as provenance for the preserved upstream-protocol result.

The four protocol IDs are `legacy_leak__legacy_trace`, `train_only__legacy_trace`, `legacy_leak__train_trace`, and `train_only__train_trace`. For QSVM, `legacy_trace` means square-only trace scaling; `train_trace` applies the training trace to both the training and matching cross-kernel. For MerLin, the common launcher label `legacy_trace` maps to no normalization. It is only a shared protocol label and does not imply that MerLin inherits any behavior from the imported repository.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

results_dir = Path.cwd() / 'results'
summary_path = results_dir / 'protocol_matrix_n500_q4_q6' / 'protocol_summary.csv'
if not summary_path.is_file():
    raise FileNotFoundError('Run this notebook from the qsvm_medimage directory.')

protocol_summary = pd.read_csv(summary_path)
expected_protocols = {
    'legacy_leak__legacy_trace',
    'train_only__legacy_trace',
    'legacy_leak__train_trace',
    'train_only__train_trace',
}
if len(protocol_summary) != 32:
    raise ValueError('Expected 32 aggregate protocol rows.')
if set(protocol_summary['protocol_id']) != expected_protocols:
    raise ValueError('Unexpected or incomplete protocol IDs.')
if set(protocol_summary['q']) != {4, 6}:
    raise ValueError('Expected q=4 and q=6 only.')
if set(protocol_summary['model']) != {'qsvm', 'merlin_fidelity'}:
    raise ValueError('Unexpected or incomplete quantum models.')
if set(protocol_summary['baseline']) != {'linear_c1', 'rbf_tuned'}:
    raise ValueError('Unexpected or incomplete baselines.')
print('Validated: 32 aggregate rows from the completed protocol matrix.')
protocol_summary.head()

Validated: 32 aggregate rows from the completed protocol matrix.


,protocol_id,preprocessing_protocol,trace_protocol,kernel_normalization,q,model,baseline,mean_f1,std_f1,baseline_mean_f1,baseline_std_f1,delta_f1,wins,ties,losses,seeds
0,legacy_leak__legacy_trace,legacy_train_plus_heldout,NaN,none,4,merlin_fidelity,linear_c1,0.757814,0.131587,0.758009,0.079520,-0.000195,6,0,4,10
1,legacy_leak__legacy_trace,legacy_train_plus_heldout,NaN,none,4,merlin_fidelity,rbf_tuned,0.757814,0.131587,0.759963,0.121262,-0.002150,4,2,4,10
2,legacy_leak__legacy_trace,legacy_train_plus_heldout,NaN,none,6,merlin_fidelity,linear_c1,0.770021,0.096427,0.786008,0.094895,-0.015987,2,3,5,10
3,legacy_leak__legacy_trace,legacy_train_plus_heldout,NaN,none,6,merlin_fidelity,rbf_tuned,0.770021,0.096427,0.818913,0.090559,-0.048891,1,1,8,10
4,legacy_leak__legacy_trace,legacy_train_plus_heldout,legacy_square_only,NaN,4,qsvm,linear_c1,0.808947,0.059340,0.758009,0.079520,0.050938,7,0,3,10


## 3. QSVM protocol sensitivity

The table presents all four QSVM protocols at q=4 and q=6. Means, sample standard deviations, deltas, and paired W/T/L counts are read from the committed aggregate summary. MerLin remains separate in Section 4.

**W/T/L = Wins / Ties / Losses**, counted seed by seed from the first-named model's perspective. It measures paired seed-level stability and is not the paper's count of model–qubit configurations.

In [2]:
qsvm_rows = protocol_summary.query("model == 'qsvm'").copy()
identity = [
    'protocol_id', 'preprocessing_protocol', 'trace_protocol',
    'q', 'mean_f1', 'std_f1',
]
comparison = [
    'baseline_mean_f1', 'baseline_std_f1', 'delta_f1',
    'wins', 'ties', 'losses',
]
linear_rows = (
    qsvm_rows.query("baseline == 'linear_c1'")[identity + comparison]
    .rename(columns={column: f'linear_{column}' for column in comparison})
)
rbf_rows = (
    qsvm_rows.query("baseline == 'rbf_tuned'")[identity + comparison]
    .rename(columns={column: f'rbf_{column}' for column in comparison})
)
qsvm_table = linear_rows.merge(rbf_rows, on=identity, validate='one_to_one')
display(qsvm_table.round(6))

,protocol_id,preprocessing_protocol,trace_protocol,q,mean_f1,std_f1,linear_baseline_mean_f1,linear_baseline_std_f1,linear_delta_f1,linear_wins,linear_ties,linear_losses,rbf_baseline_mean_f1,rbf_baseline_std_f1,rbf_delta_f1,rbf_wins,rbf_ties,rbf_losses
0,legacy_leak__legacy_trace,legacy_train_plus_heldout,legacy_square_only,4,0.808947,0.059340,0.758009,0.079520,0.050938,7,0,3,0.759963,0.121262,0.048984,6,1,3
1,legacy_leak__legacy_trace,legacy_train_plus_heldout,legacy_square_only,6,0.822105,0.077912,0.786008,0.094895,0.036097,7,0,3,0.818913,0.090559,0.003193,4,1,5
2,legacy_leak__train_trace,legacy_train_plus_heldout,train_trace,4,0.000000,0.000000,0.758009,0.079520,-0.758009,0,0,10,0.759963,0.121262,-0.759963,0,0,10
3,legacy_leak__train_trace,legacy_train_plus_heldout,train_trace,6,0.000000,0.000000,0.786008,0.094895,-0.786008,0,0,10,0.818913,0.090559,-0.818913,0,0,10
4,train_only__legacy_trace,train_only,legacy_square_only,4,0.813727,0.057270,0.758009,0.079520,0.055718,8,0,2,0.759963,0.121262,0.053763,6,1,3
5,train_only__legacy_trace,train_only,legacy_square_only,6,0.822105,0.077912,0.786008,0.094895,0.036097,7,0,3,0.814960,0.086978,0.007145,4,1,5
6,train_only__train_trace,train_only,train_trace,4,0.000000,0.000000,0.758009,0.079520,-0.758009,0,0,10,0.759963,0.121262,-0.759963,0,0,10
7,train_only__train_trace,train_only,train_trace,6,0.000000,0.000000,0.786008,0.094895,-0.786008,0,0,10,0.814960,0.086978,-0.814960,0,0,10


For the QSVM tables:

- `linear_delta_f1 = QSVM F1 - linear SVM F1`;
- `rbf_delta_f1 = QSVM F1 - tuned RBF SVM F1`.

Positive values favor the QSVM.

**W/T/L = Wins / Ties / Losses**, counted seed by seed from the QSVM perspective. These counts are not the paper's model–qubit-configuration wins.

### Preserved upstream-protocol result

In [3]:
preserved_upstream = qsvm_table.query(
    "protocol_id == 'legacy_leak__legacy_trace'"
).copy()
display(preserved_upstream.round(6))

,protocol_id,preprocessing_protocol,trace_protocol,q,mean_f1,std_f1,linear_baseline_mean_f1,linear_baseline_std_f1,linear_delta_f1,linear_wins,linear_ties,linear_losses,rbf_baseline_mean_f1,rbf_baseline_std_f1,rbf_delta_f1,rbf_wins,rbf_ties,rbf_losses
0,legacy_leak__legacy_trace,legacy_train_plus_heldout,legacy_square_only,4,0.808947,0.059340,0.758009,0.079520,0.050938,7,0,3,0.759963,0.121262,0.048984,6,1,3
1,legacy_leak__legacy_trace,legacy_train_plus_heldout,legacy_square_only,6,0.822105,0.077912,0.786008,0.094895,0.036097,7,0,3,0.818913,0.090559,0.003193,4,1,5


**W/T/L = Wins / Ties / Losses**, counted seed by seed from the QSVM perspective. These counts are not the paper's model–qubit-configuration wins.

Delta columns use the same `QSVM F1 - baseline F1` convention.

In [4]:
print('No figure is generated by the protocol-audit notebook.')

No figure is generated by the protocol-audit notebook.


### Main audit finding

- Changing from leaky to train-only MinMax fitting changes every matching aggregate minority-F1 result by less than 0.005. The local empirical effect is negligible, but using held-out extrema remains leakage.
- With consistent QSVM train-trace scaling, minority F1 is zero on every paired seed at q=4 and q=6 under both preprocessing variants.
- The square training kernel is trace-normalized in both QSVM trace variants. The controlled difference is whether the held-out cross-kernel is divided by the same training trace. The favorable held-out result of the preserved upstream path therefore depends on this scale mismatch in this surrogate.
- The local linear baseline has non-zero minority F1, so the paper's linear-collapse-avoidance mechanism is not reproduced.

The positive QSVM rows displayed separately above correspond only to `legacy_leak__legacy_trace`: leaky MinMax fitting plus square-only QSVM trace scaling. They are retained for provenance and are not the main protocol-sensitivity result.

## 4. MerLin photonic adaptation

For every q–seed pair, MerLin receives the same N=500 subset, 400/50/50 split, and PCA dimension as the three Claim 1 models. The data seed varies from 0 to 9, while the MerLin circuit seed remains fixed at 0.

| Property | Matched setting |
|---|---|
| Samples | N=500 |
| Dimensions | PCA q=4 and q=6 |
| Data/split seeds | 0–9 |
| Classifier | Precomputed-kernel SVM, `C=1` |
| Photonic resources at q=4 | 5 modes, 3 photons, input `[1, 0, 1, 0, 1]` |
| Photonic resources at q=6 | 7 modes, 4 photons, input `[1, 0, 1, 0, 1, 0, 1]` |
| Circuit seed | 0, fixed |

MerLin is absent from the paper and imported upstream repository. This removes sample-and-split differences from the local comparison, but it does not make MerLin equivalent or resource-matched to the BSP QSVM. The evaluated variants are the MerLin unnormalized variant and the MerLin train-trace variant.

In [5]:
merlin_rows = protocol_summary.query("model == 'merlin_fidelity'").copy()
identity = [
    'protocol_id', 'preprocessing_protocol', 'kernel_normalization',
    'q', 'mean_f1', 'std_f1',
]
comparison = [
    'baseline_mean_f1', 'baseline_std_f1', 'delta_f1',
    'wins', 'ties', 'losses',
]
linear_rows = (
    merlin_rows.query("baseline == 'linear_c1'")[identity + comparison]
    .rename(columns={column: f'linear_{column}' for column in comparison})
)
rbf_rows = (
    merlin_rows.query("baseline == 'rbf_tuned'")[identity + comparison]
    .rename(columns={column: f'rbf_{column}' for column in comparison})
)
merlin_table = linear_rows.merge(rbf_rows, on=identity, validate='one_to_one')
display(merlin_table.round(6))

,protocol_id,preprocessing_protocol,kernel_normalization,q,mean_f1,std_f1,linear_baseline_mean_f1,linear_baseline_std_f1,linear_delta_f1,linear_wins,linear_ties,linear_losses,rbf_baseline_mean_f1,rbf_baseline_std_f1,rbf_delta_f1,rbf_wins,rbf_ties,rbf_losses
0,legacy_leak__legacy_trace,legacy_train_plus_heldout,none,4,0.757814,0.131587,0.758009,0.079520,-0.000195,6,0,4,0.759963,0.121262,-0.002150,4,2,4
1,legacy_leak__legacy_trace,legacy_train_plus_heldout,none,6,0.770021,0.096427,0.786008,0.094895,-0.015987,2,3,5,0.818913,0.090559,-0.048891,1,1,8
2,legacy_leak__train_trace,legacy_train_plus_heldout,train_trace,4,0.000000,0.000000,0.758009,0.079520,-0.758009,0,0,10,0.759963,0.121262,-0.759963,0,0,10
3,legacy_leak__train_trace,legacy_train_plus_heldout,train_trace,6,0.000000,0.000000,0.786008,0.094895,-0.786008,0,0,10,0.818913,0.090559,-0.818913,0,0,10
4,train_only__legacy_trace,train_only,none,4,0.755433,0.133619,0.758009,0.079520,-0.002576,6,0,4,0.759963,0.121262,-0.004531,4,2,4
5,train_only__legacy_trace,train_only,none,6,0.770021,0.096427,0.786008,0.094895,-0.015987,2,3,5,0.814960,0.086978,-0.044939,1,1,8
6,train_only__train_trace,train_only,train_trace,4,0.000000,0.000000,0.758009,0.079520,-0.758009,0,0,10,0.759963,0.121262,-0.759963,0,0,10
7,train_only__train_trace,train_only,train_trace,6,0.000000,0.000000,0.786008,0.094895,-0.786008,0,0,10,0.814960,0.086978,-0.814960,0,0,10


The table is written from MerLin's perspective: each delta is MerLin F1 minus the named classical baseline F1. **W/T/L = Wins / Ties / Losses**, counted seed by seed from the first-named model's perspective.

- Without trace normalization, MerLin is in the same broad performance range as the local QSVM and classical baselines.
- With train-trace normalization and fixed `C=1`, MerLin has zero minority F1 on every evaluated seed, matching the qualitative QSVM scale sensitivity.

This supports consistency of the local comparison. It does not establish a photonic or quantum advantage, an upstream issue, or an optimized MerLin result.

## 5. Conclusion and limitations

The MinMax leakage is empirically negligible in this surrogate, while consistent train-trace scaling changes the QSVM result decisively. The favorable held-out scores of the preserved upstream path depend on the train/cross-kernel scale mismatch locally. The linear model does not collapse.

These observations neither reproduce nor refute the paper's MIMIC-CXR claim. The local experiment changes the dataset, target, representation, sample count, and q range. It covers only q=4 and q=6, only 50 test samples per seed, subset/split rather than embedding seeds, and no local paired significance test.

MerLin is a separate photonic adaptation, is not resource-matched to the BSP circuit, and has not had its feature map or `C` optimized. The audit establishes no author intent and cannot validate or invalidate the original MIMIC-CXR results.